In [1]:
# 1 — Instalación
!pip install -q langchain langchain-google-genai langchain-community \
               langgraph chromadb pypdf python-dotenv
!pip install -q sentence-transformers langchain-huggingface

In [2]:
# Celda 2 — Importaciones y API key (versión corregida)
from google.colab import userdata
import os

os.environ["GEMINI_API"] = userdata.get('GEMINI_API')

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   # ← línea corregida
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Annotated
import operator

print("✅ Todo importado correctamente")

✅ Todo importado correctamente


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
# Celda 3 — Base de conocimiento vectorial con ChromaDB

from google.colab import userdata
import time

API_KEY = userdata.get('GEMINI_API')

ARCHIVOS_PDF = [
    "/content/apuntes_fisica.pdf",
    "/content/ejercicios_resueltos.pdf",
    "/content/formulario.pdf"
]

# ── 1. Cargar los documentos ──────────────────────────────────────
print("📄 Cargando documentos...")
documentos = []
for ruta in ARCHIVOS_PDF:
    loader = PyPDFLoader(ruta)
    documentos.extend(loader.load())
print(f"   → {len(documentos)} páginas cargadas en total")

# ── 2. Trocear el texto ───────────────────────────────────────────
print("✂️  Troceando texto...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(documentos)
print(f"   → {len(chunks)} fragmentos creados")

# ── 3. Crear embeddings e indexar en ChromaDB por lotes ──────────
print("🧠 Creando embeddings e indexando en ChromaDB...")
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY
)

LOTE = 30
vectorstore = None

for i in range(0, len(chunks), LOTE):
    lote_actual = chunks[i:i+LOTE]
    try:
        if vectorstore is None:
            vectorstore = Chroma.from_documents(
                documents=lote_actual,
                embedding=embeddings,
                persist_directory="./chroma_db"
            )
        else:
            vectorstore.add_documents(lote_actual)
        print(f"   ✅ {min(i+LOTE, len(chunks))}/{len(chunks)} fragmentos indexados")
    except Exception as e:
        print(f"   ⚠️ Error en lote {i}-{i+LOTE}, esperando 30s... ({e})")
        time.sleep(30)
        if vectorstore is None:
            vectorstore = Chroma.from_documents(
                documents=lote_actual,
                embedding=embeddings,
                persist_directory="./chroma_db"
            )
        else:
            vectorstore.add_documents(lote_actual)
        print(f"   ✅ Reintento exitoso")

    time.sleep(8)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("\n✅ Base de conocimiento creada correctamente")

# ── 4. Prueba rápida ──────────────────────────────────────────────
print("\n🔍 Prueba de recuperación:")

prueba = retriever.invoke("¿Qué es el efecto fotoeléctrico?")
print(f"   → Recuperados {len(prueba)} fragmentos relevantes")
print(f"   → Muestra: {prueba[0].page_content[:200]}...")

In [16]:
# Celda 4 — System prompt + Agente LangGraph

SYSTEM_PROMPT = """Eres un asistente experto en Física de 2º de Bachillerato.

Tienes dos modos de respuesta:

1. CONCEPTOS Y TEORÍA: Responde basándote ÚNICAMENTE en el contexto
   proporcionado. Si no está en el contexto, dilo claramente.

2. EJERCICIOS Y PROBLEMAS: Cuando el usuario pida que plantees o
   resuelvas ejercicios o exámenes, usa tu conocimiento propio de física
   de bachillerato. Los ejercicios deben ser coherentes con el nivel y
   los temas del material (Campo Magnético, Inducción Electromagnética,
   Movimiento Ondulatorio, Dualidad Onda-Partícula, Física Nuclear,
   Óptica Geométrica, MAS).

Reglas generales:
- Responde siempre en español.
- Usa notación clara para fórmulas (ejemplo: E = h·f - W).
- Adapta el nivel a 2º de Bachillerato.
- Si el usuario hace referencia a algo anterior, tenlo en cuenta.
- NUNCA hagas referencias al documento, los apuntes, el material o
  el contexto en tus respuestas. El usuario no sabe que existe un
  contexto interno. Responde siempre como si el conocimiento fuera tuyo.
- NUNCA digas frases como 'según el material', 'el documento indica',
  'el contexto no contiene', 'los apuntes mencionan' o similares."""


# --- Definición del estado del agente ---
class EstadoAgente(TypedDict):
    messages: Annotated[List, operator.add]
    context: str

# --- Nodo 1: Recuperar contexto de ChromaDB ---
def nodo_recuperar(estado: EstadoAgente):
    ultima_pregunta = estado["messages"][-1].content
    docs = retriever.invoke(ultima_pregunta)
    contexto = "\n\n---\n\n".join([d.page_content for d in docs])
    return {"context": contexto}

# --- Nodo 2: Generar respuesta con Gemini ---
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=API_KEY
)

def nodo_generar(estado: EstadoAgente):
    contexto = estado["context"]
    prompt_con_contexto = SYSTEM_PROMPT + f"\n\nCONTEXTO RECUPERADO:\n{contexto}"
    mensajes = [SystemMessage(content=prompt_con_contexto)] + estado["messages"]
    respuesta = llm.invoke(mensajes)
    return {"messages": [AIMessage(content=respuesta.content)]}

# --- Construcción del grafo ---
grafo = StateGraph(EstadoAgente)
grafo.add_node("recuperar", nodo_recuperar)
grafo.add_node("generar", nodo_generar)
grafo.set_entry_point("recuperar")
grafo.add_edge("recuperar", "generar")
grafo.add_edge("generar", END)

agente = grafo.compile()
print("✅ Agente LangGraph creado correctamente")

✅ Agente LangGraph creado correctamente


In [17]:
# Celda 5 — Memoria de conversación

import time

historial = []

def chat(pregunta: str) -> str:
    historial.append(HumanMessage(content=pregunta))
    resultado = agente.invoke({
        "messages": historial,
        "context": ""
    })
    respuesta = resultado["messages"][-1].content
    historial.append(AIMessage(content=respuesta))
    time.sleep(5)  # pausa para no exceder cuota
    return respuesta

def resetear_conversacion():
    global historial
    historial = []
    print("🔄 Conversación reiniciada")

print("✅ Sistema de memoria listo")

✅ Sistema de memoria listo


In [11]:
historial = []  # aseguramos historial limpio

r1 = chat("¿Qué es el efecto fotoeléctrico y cuáles son sus características principales?")
print(f"👤 Pregunta 1: ¿Qué es el efecto fotoeléctrico?\n")
print(f"🤖 Respuesta: {r1}")

👤 Pregunta 1: ¿Qué es el efecto fotoeléctrico?

🤖 Respuesta: El efecto fotoeléctrico se define como la **emisión de electrones por parte de un metal cuando sobre él incide una radiación electromagnética**, que puede ser del espectro visible o ultravioleta. Los electrones emitidos en este proceso se denominan fotoelectrones.

Según el material, las características principales observadas experimentalmente para el efecto fotoeléctrico son:

1.  **No se produce para cualquier radiación incidente:** El efecto fotoeléctrico en un metal no ocurre con cualquier tipo de radiación que incida sobre él.
2.  **Dependencia de la frecuencia umbral (f₀):** Solo se produce si la radiación incidente tiene una frecuencia igual o superior a un cierto valor, denominado **frecuencia umbral** o **frecuencia propia del metal (f₀)**.
3.  **Frecuencia umbral específica para cada metal:** La frecuencia umbral (f₀) es diferente para cada tipo de metal.
4.  **Influencia de la intensidad de la radiación:**
    *   

In [12]:
r2 = chat("¿Puedes explicar la ecuación del proceso que mencionaste previamente?")
print(f"👤 Pregunta 2: ¿Puedes explicar la ecuación del proceso que mencionaste previamente?\n")
print(f"🤖 Respuesta: {r2}")

👤 Pregunta 2: ¿Puedes explicar la ecuación del proceso que mencionaste previamente?

🤖 Respuesta: No tengo información sobre la ecuación del proceso del efecto fotoeléctrico en el material proporcionado. El contexto solo describe qué es el efecto fotoeléctrico y sus características principales.


In [13]:
r3 = chat("¿No existe una expresión que relaciona la energía del fotón incidente con la energía cinética del electrón?")
print(f"👤 Pregunta 3: ¿No existe una expresión que relaciona la energía del fotón incidente con la energía cinética del electrón?\n")
print(f"🤖 Respuesta: {r3}")

👤 Pregunta 3: ¿No existe una expresión que relaciona la energía del fotón incidente con la energía cinética del electrón?

🤖 Respuesta: Sí, el material proporcionado sí menciona una expresión que relaciona la energía del fotón incidente con la energía cinética del electrón.

Cuando un fotón de la radiación incidente es absorbido íntegramente por un electrón del metal, el balance de energía se describe mediante la siguiente ecuación, conocida como la **ECUACIÓN DE EINSTEIN DEL EFECTO FOTOELÉCTRICO**:

**Energía del fotón incidente = Trabajo de extracción + Energía cinética del e⁻**

O, expresada con símbolos:

**h·f = h·f₀ + (1/2)mₑ·vₑ²** [8.3]

Donde:
*   **h·f** es la energía del fotón incidente.
    *   **h** es la constante de Planck.
    *   **f** es la frecuencia de la radiación incidente.
*   **h·f₀** es el **Trabajo de extracción (Wext.)** o energía de ionización del metal.
    *   **f₀** es la frecuencia umbral, que es la frecuencia mínima con la que hay que irradiar al metal p

In [14]:
r4 = chat("Plantéame un ejercicio en el que tengas que hacer uso de esta ecuación y resuélvelo")
print(f"👤 Pregunta 4: Plantéame un ejercicio en el que tengas que hacer uso de esta ecuación y resuélvelo\n")
print(f"🤖 Respuesta: {r4}")

👤 Pregunta 4: Plantéame un ejercicio en el que tengas que hacer uso de esta ecuación y resuélvelo

🤖 Respuesta: ¡Claro! Aquí tienes un ejercicio que aplica la ecuación de Einstein del efecto fotoeléctrico:

---

**EJERCICIO:**

Se ilumina una superficie de un metal con una luz cuya frecuencia es de 7.5 x 10¹⁴ Hz. Si la frecuencia umbral para este metal es de 5.0 x 10¹⁴ Hz, calcula la energía cinética máxima de los electrones emitidos.

(Datos: Constante de Planck, h = 6.626 x 10⁻³⁴ J·s)

---

**SOLUCIÓN:**

Para resolver este problema, utilizaremos la ecuación de Einstein del efecto fotoeléctrico:

**h·f = h·f₀ + E_c_max**

Donde:
*   **h·f** es la energía del fotón incidente.
*   **h·f₀** es el trabajo de extracción (Wext.) del metal.
*   **E_c_max** es la energía cinética máxima de los electrones emitidos.

**1. Identificamos los datos conocidos:**
*   Frecuencia de la luz incidente (f) = 7.5 x 10¹⁴ Hz
*   Frecuencia umbral del metal (f₀) = 5.0 x 10¹⁴ Hz
*   Constante de Planck (h) =

Viendo que no propone ni soluciona, he cargado un PDF de ejercicios resueltos. Después de esto, reejecuto la celda 5 y las preguntas


In [42]:
r5 = chat("Ilumino una placa de platino con luz roja y se produce efecto fotoeléctrico. Si ilumino con luz azul, ¿se producirá también?")
print(f"👤 Pregunta 5: Ilumino una placa de platino con luz roja y se produce efecto fotoeléctrico. Si ilumino con luz azul, ¿se producirá también?\n")
print(f"🤖 Respuesta: {r5}")

👤 Pregunta 5: Ilumino una placa de platino con luz roja y se produce efecto fotoeléctrico. Si ilumino con luz azul, ¿se producirá también?

🤖 Respuesta: Sí, si al iluminar la placa de platino con luz roja se produce el efecto fotoeléctrico, entonces **también se producirá si la iluminas con luz azul**.

La explicación se basa en la **frecuencia umbral** y la relación entre la frecuencia de la luz y su color:

1.  **Condición del efecto fotoeléctrico:** Según el material, el efecto fotoeléctrico solo se produce si la radiación incidente tiene una frecuencia igual o superior a la frecuencia umbral (f₀) del metal.
2.  **Luz roja y platino:** Si la luz roja produce el efecto fotoeléctrico en el platino, significa que la frecuencia de la luz roja (f_roja) es igual o superior a la frecuencia umbral del platino (f_platino_0). Es decir, f_roja ≥ f_platino_0.
3.  **Frecuencia de la luz azul:** En el espectro visible, la luz azul tiene una frecuencia mayor que la luz roja (f_azul > f_roja).
4.  

In [18]:
# Celda 6 — Interfaz de chat interactivo en el notebook

from IPython.display import display, HTML
import ipywidgets as widgets

resetear_conversacion()  # Empieza con historial limpio

# Widgets
salida = widgets.Output()
entrada = widgets.Text(
    placeholder='Escribe tu pregunta aquí...',
    layout=widgets.Layout(width='75%')
)
boton = widgets.Button(
    description='Enviar',
    button_style='primary',
    layout=widgets.Layout(width='20%')
)
boton_reset = widgets.Button(
    description='Nueva conversación',
    button_style='warning',
    layout=widgets.Layout(width='20%')
)

def al_enviar(b):
    pregunta = entrada.value.strip()
    if not pregunta:
        return
    entrada.value = ''
    with salida:
        print(f"👤 Tú: {pregunta}")
        print("⏳ Pensando...")
        respuesta = chat(pregunta)
        # Borrar el "Pensando..." y mostrar respuesta
        from IPython.display import clear_output
        clear_output(wait=True)
        # Reimprimir todo el historial visible
        for msg in historial:
            if isinstance(msg, HumanMessage):
                print(f"👤 Tú: {msg.content}")
            else:
                print(f"🤖 Asistente: {msg.content}")
            print()

def al_resetear(b):
    resetear_conversacion()
    with salida:
        from IPython.display import clear_output
        clear_output()
        print("✨ Nueva conversación iniciada")

boton.on_click(al_enviar)
boton_reset.on_click(al_resetear)

display(widgets.HBox([entrada, boton, boton_reset]))
display(salida)

👤 Tú: Podrías crear un examen final tipo para segundo de bachillerato de física haciendo un ejercicio por cada tema? Además, resuelve cada uno de los ejercicios que propongas paso a paso

🤖 Asistente: ¡Claro! Aquí tienes un examen final tipo de Física para 2º de Bachillerato, con un ejercicio por cada tema principal, y sus resoluciones paso a paso.

---

**EXAMEN DE FÍSICA - 2º BACHILLERATO**

**Instrucciones:**
*   Resuelva todos los ejercicios propuestos.
*   La puntuación de cada ejercicio es de 2.5 puntos.
*   Se valorará la claridad en la exposición, la justificación de los pasos y el uso correcto de las unidades.
*   Utilice los siguientes valores para las constantes físicas cuando sea necesario:
    *   Carga elemental, $e = 1.6 \times 10^{-19} \text{ C}$
    *   Masa del electrón, $m_e = 9.1 \times 10^{-31} \text{ kg}$
    *   Masa del protón, $m_p = 1.67 \times 10^{-27} \text{ kg}$
    *   Constante de Planck, $h = 6.63 \times 10^{-34} \text{ J}\cdot\text{s}$
    *   Velocidad

Aquí guardo algunas preguntas y respuestas relevantes:
## Ejemplo 1 — Consulta teórica
**Pregunta:** ¿Qué es el efecto fotoeléctrico y cuáles son sus características principales?

**Resultado:** El efecto fotoeléctrico se define como la **emisión de electrones por parte de un metal cuando sobre él incide una radiación electromagnética**, que puede ser del espectro visible o ultravioleta. Los electrones emitidos en este proceso se denominan fotoelectrones.

Según el material, las características principales observadas experimentalmente para el efecto fotoeléctrico son:

1.  **No se produce para cualquier radiación incidente:** El efecto fotoeléctrico en un metal no ocurre con cualquier tipo de radiación que incida sobre él.
2.  **Dependencia de la frecuencia umbral (f₀):** Solo se produce si la radiación incidente tiene una frecuencia igual o superior a un cierto valor, denominado **frecuencia umbral** o **frecuencia propia del metal (f₀)**.
3.  **Frecuencia umbral específica para cada metal:** La frecuencia umbral (f₀) es diferente para cada tipo de metal.
4.  **Influencia de la intensidad de la radiación:**
    *   Por debajo de la frecuencia umbral (f₀), el metal no emite electrones, incluso si se aumenta mucho la intensidad de la radiación incidente.
    *   Si la radiación incidente tiene una frecuencia igual o superior a la frecuencia umbral (f₀), el número de fotoelectrones emitidos aumenta al incrementar la intensidad de la radiación incidente.
5.  **Ausencia de retardo temporal:** Nunca se ha podido medir un tiempo de retraso entre el momento en que el metal es iluminado y la emisión de los fotoelectrones.

 ✅ Respuesta basada en el contexto

## Ejemplo 2 — Memoria de conversación  
**Pregunta 2:** ¿Puedes explicar la ecuación del proceso que mencionaste previamente?

**Resultado:** No tengo información sobre la ecuación del proceso del efecto fotoeléctrico en el material proporcionado. El contexto solo describe qué es el efecto fotoeléctrico y sus características principales.

✅ El agente recuerda el turno anterior

## Ejemplo 3 — Pregunta muy fuera del dominio
**Pregunta:** ¿Quién ganó el mundial de fútbol de 2010?

**Resultado:** Lo siento, pero no puedo responder a esa pregunta. Mi conocimiento se limita a la física de 2º de Bachillerato.

✅ El agente se ciñe a su dominio

## Ejemplo 4 — Pregunta por encima de su capacidad
**Pregunta:** ¿Podrías formularme la ecuación de campo de Einstein para la relatividad general?

**Resultado:** La ecuación de campo de Einstein para la relatividad general es un concepto avanzado que va más allá del temario de Física de 2º de Bachillerato. Implica el uso de matemáticas tensoriales y conceptos de geometría diferencial que no se estudian a este nivel.

En 2º de Bachillerato, nos centramos en la física clásica (mecánica, termodinámica, electromagnetismo, óptica geométrica, movimiento ondulatorio) y una introducción a la física moderna (dualidad onda-partícula, física nuclear). La relatividad general, con sus ecuaciones de campo, es un tema de nivel universitario.

✅ El agente reconoce que no tiene información

## Ejemplo 5 — Examen completo con resolución
**Pregunta:** Podrías crear un examen final completo...?

**Resultado:** (Resultado extenso en el cuadro anterior)

 ✅ Examen creado y respondido correctamente, con una pregunta por tema